[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/python-fundamentals-hh/blob/main/notebooks/09_naip-imagery/09_01_find_download_naip.ipynb)

# Module 9, Lesson 1: Find, Download, and Inspect NAIP Aerial Imagery
## Getting a trustworthy, right-sized aerial image for your study area

### Welcome!

Aerial imagery is usually the first thing an H&H engineer opens on a new project. Before you delineate anything or build a model, you want to see the site: where the roads and rooftops are, where water sits, and where the ground has been graded. The National Agriculture Imagery Program (NAIP) gives you that view for free across the lower 48 states at roughly 0.6 to 1 meter resolution.

In this lesson you'll find NAIP imagery for a small area inside the Town of South Greeley watershed near Cheyenne, Wyoming. That's the same HUC-12 used in Modules 3, 4, 5, and 8. You'll check which acquisitions actually cover the area, download only the part you need, and confirm the file is what you think it is before you use it.

The area includes a community college campus with two ponds, an athletic field, parking lots, and a new subdivision that was still under construction when the 2022 imagery was flown. We'll come back to that subdivision in Lessons 2 and 3.

### What You'll Work Through Today

- Tell the difference between aerial and satellite imagery, and between a catalog record and the image file it points to
- Search the NAIP catalog for a small area of interest (AOI)
- Work politely with a free shared service, including what to do when it rate limits you
- Build an inventory of every acquisition, with dates, resolution, and how much of the AOI each one covers
- Pick an image that covers the *whole* AOI, not just part of it
- Estimate the download size before transferring anything
- Read only the AOI from a cloud-hosted image and save it as a GeoTIFF
- Check CRS, bounds, bands, resolution, and valid pixels before trusting the file
- Take a first look at the red, green, blue, and near-infrared bands
- Find and fix a selection bug that quietly produces an incomplete image

### Module 9 at a Glance

| Lesson | Question | Main output |
|---|---|---|
| **1 (this one)** | How do I get a trustworthy image for my area? | 2022 four-band GeoTIFF, source inventory, quicklook |
| 2 | What changed between two acquisitions, and what only *looks* like it changed? | 2012 and 2022 images on a common grid, comparison figure, observation log |
| 3 | Can a handful of labeled examples produce a useful land cover screening map? | Classified GeoTIFF, validation table, class areas |

Each lesson runs on its own in a fresh Colab session. If you skip one, the next lesson downloads the course copy of whatever it needs.

### Lesson Structure

1. **Mental Model** - Aerial imagery, catalog records, footprints, and bands
2. **Workspace Setup** - Packages and lesson settings
3. **Watershed and AOI** - Why we don't start with the whole watershed
4. **Catalog Search** - Find NAIP items that touch the AOI
5. **Choose an Acquisition** - Coverage first, then date
6. **Size Planning** - Estimate before you download
7. **Download** - Read just the AOI from a cloud-hosted file
8. **Inspect** - Check the file before you trust it
9. **First Look** - True color and the four individual bands
10. **Export** - A token-free source inventory
11. **Practice Debugging** - When "the newest image" isn't the right image

## Lesson Information

**Lesson Level:** Intermediate

**Purpose:** Get a small, well-documented NAIP image for a study area, and learn the checks that separate a usable image from one that only looks usable.

### Learning Objectives

By the end of this lesson, you'll be able to:

- Explain why NAIP is aerial imagery and what that means for its dates and resolution.
- Distinguish a STAC search result (a catalog record) from the image asset it points to.
- Explain why an image that *intersects* your AOI may not *cover* it.
- Recognize an HTTP 429 rate limit and handle it with retries and backoff instead of treating it as a code error.
- Identify acquisition date, footprint, ground sample distance (GSD), band order, CRS, and valid pixels before interpreting an image.
- Read a bounded window from a Cloud Optimized GeoTIFF and save it with provenance.

### Prerequisites

- Module 3 (vector data and coordinate reference systems)
- Module 4 (raster basics: CRS, resolution, transform, NoData)
- Comfort with Python functions, lists, dictionaries, and pandas tables

Module 8 is useful background because Lesson 3 compares its results with Annual NLCD, but it isn't required.

### Inputs and Outputs

| | File | Notes |
|---|---|---|
| Input | `NHD__Watershed_Boundaries_HUC_12_Selected.zip` | Found locally or downloaded from the course repository automatically |
| Input | Live NAIP catalog and imagery | Microsoft Planetary Computer; free, no account or API key |
| Output | `module9_outputs/naip_2022_teaching_aoi.tif` | Four-band GeoTIFF of the teaching AOI |
| Output | `module9_outputs/naip_source_inventory.csv` | Every acquisition that touches the AOI, with coverage |
| Output | `module9_outputs/naip_2022_quicklook.png` | True-color preview |

Everything runs on a standard Colab CPU runtime. You don't need a GPU, a cloud account, Google Drive, or local GIS software. If the live service is down, the notebook uses a matching course copy of the image and tells you it did.

## Using AI in This Lesson

An AI assistant is most useful here when you give it something specific to reason about: a row from the inventory table, a failed check, or a printed raster profile. Ask it to explain the evidence, not to write the workflow for you.

Some questions worth trying as you go:

- *"Here's my NAIP inventory table. Which columns tell me whether one image can cover my whole AOI by itself?"*
- *"My raster's pixel size is 0.6 m but the file is 1,834 columns wide. Walk me through how that relates to the AOI width."*
- *"What's the difference between a STAC item and its image asset? Why does the notebook sign the asset but not the item?"*

Two rules for this lesson:

1. Check what the assistant tells you against the printed metadata. If it says the image is 1 m resolution and the notebook prints 0.6 m, believe the notebook.
2. Never paste a signed image URL into a prompt, message, or issue report. It contains a temporary access token. The notebook never prints one, and you shouldn't need one to ask a good question.

## Part 1: Mental Model

### Aerial, Not Satellite

NAIP imagery is flown from aircraft during the agricultural growing season, usually once every two to three years per state. That has practical consequences:

- **Resolution is fine.** Recent Wyoming acquisitions are 0.6 m; older ones are 1 m. Satellite products like Landsat are 30 m.
- **Dates vary.** One state's "2022 NAIP" can be flown over several weeks. Neighboring tiles can have different flight dates, so the acquisition date belongs to each image, not to the year.
- **It's leaf-on imagery.** Trees and grass are at or near full cover, which helps with vegetation but can hide channels, culverts, and ground conditions.
- **It's not calibrated reflectance.** NAIP pixel values are 8-bit (0 to 255) and are adjusted for appearance. They're great for seeing and mapping, but the same surface can have different values in different years.

### A Catalog Card and the File It Points To

Microsoft Planetary Computer hosts NAIP as a searchable STAC catalog. STAC stands for SpatioTemporal Asset Catalog. Think of it like a library:

| Term | Library analogy | What it is here |
|---|---|---|
| Collection | The shelf | All NAIP imagery (`naip`) |
| Item | A catalog card | One image tile: footprint, date, GSD, CRS |
| Asset | The book | The actual four-band Cloud Optimized GeoTIFF |
| Signing | Checking the book out | Adding a short-lived access token to the asset URL |

Searching returns catalog cards. No image data moves until you open an asset. A Cloud Optimized GeoTIFF (COG) is laid out so you can read just one small window of it over the internet, which is how we'll download a 1.7 km² area without transferring the whole tile it sits in.

### Intersects Is Not the Same as Covers

A catalog search returns every item whose footprint **touches** your AOI. Touching isn't enough:

```
 Item footprint              Item footprint
+--------------+            +---------+
|  +--------+  |            |      +--+------+
|  |  AOI   |  |            |      |AO|I    |
|  +--------+  |            |      +--+------+
+--------------+            +---------+
   covers: yes                intersects: yes, covers: no
```

The right-hand case is a trap, and we'll fall into it on purpose in Part 11.

### Two Different Resolutions

- **Ground sample distance (GSD)** is a property of the source image: how much ground one source pixel represents.
- **Output pixel size** is whatever grid you write your file on.

In this lesson they're the same because we read the source pixels without resampling. In Lesson 2 they won't be, and it'll matter.

### The Four Bands

NAIP stores four bands in this order: **Red, Green, Blue, Near infrared (NIR)**. Healthy vegetation reflects a lot of NIR, so it looks bright in that band. Water absorbs NIR and looks very dark. We'll use that in all three lessons.

## Part 2: Workspace Setup

### Install What's Missing

Colab already includes most of the packages this lesson needs. The next cell checks for each one and installs only what's missing. On a fresh Colab runtime that's usually `pystac-client` and `planetary-computer`, which takes under a minute.

In [ ]:
import importlib.util
import subprocess
import sys

# Map the name you import to the name pip installs
REQUIRED_PACKAGES = {
    'pystac_client': 'pystac-client',
    'planetary_computer': 'planetary-computer',
    'rasterio': 'rasterio',
    'geopandas': 'geopandas',
}

missing = [
    pip_name for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    print(f"Installing: {', '.join(missing)}")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
    print("Done.")
else:
    print("All required packages are already installed.")

### Import the Libraries

The imports are grouped by job. The last lines print package versions. If something behaves differently from what the lesson describes, the versions are the first thing to compare.

In [ ]:
# Standard library
import datetime
import math
import shutil
from pathlib import Path

# Tables, arrays, and web requests
import numpy as np
import pandas as pd
import requests

# Vector data
import geopandas as gpd
from shapely.geometry import box, shape

# Raster data
import rasterio
import rasterio.errors
from rasterio.plot import plotting_extent
from rasterio.windows import Window, from_bounds

# NAIP catalog search and temporary access
import pystac_client
import planetary_computer
from pystac_client.stac_api_io import StacApiIO
from urllib3.util.retry import Retry

# Plotting
import matplotlib.pyplot as plt
from matplotlib import font_manager

# Every figure in this lesson uses Arial. Colab runs on Linux, which does not
# ship Arial, so the list falls back to Liberation Sans: a font drawn with the
# same letter widths and shapes, made as a drop-in replacement. Naming a font
# that does exist is also what keeps matplotlib from printing a warning for
# every label it draws.
FONT_PREFERENCE = ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = FONT_PREFERENCE

installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
active_font = next((name for name in FONT_PREFERENCE if name in installed_fonts), None)
print(f"Figure font: {active_font or 'matplotlib default'}")

for package in (np, pd, gpd, rasterio, pystac_client):
    print(f"{package.__name__:<15} {package.__version__}")

### Set the Lesson Controls

All the settings live in one cell so you can adapt the lesson without hunting through the notebook.

- `AOI_BOUNDS` is the teaching area: a 1.1 km by 1.55 km rectangle inside the watershed. The corners are in **NAD83 / UTM zone 13N** meters (EPSG:26913), which is the CRS the NAIP source files use here. Using the source CRS means we can read pixels without reprojecting them.
- `REFERENCE_YEAR = 2022` was chosen after checking the catalog while this lesson was being written. The notebook still verifies that choice live instead of assuming it.
- `MINIMUM_COVERAGE_PCT` is how much of the AOI one image must cover to be used on its own. We use 99.9 percent, which is effectively "all of it."
- `COURSE_DATA_URL` is the fallback location for course copies of the data.

In [ ]:
# The watershed used throughout the course
TARGET_HUC = '101900090108'   # Town of South Greeley, Cheyenne, WY

# Teaching AOI corners in NAD83 / UTM zone 13N meters
AOI_CRS = 'EPSG:26913'
AOI_BOUNDS = (517750, 4549350, 518850, 4550900)   # west, south, east, north

# Acquisition year checked before the lesson was written
REFERENCE_YEAR = 2022

# One image must cover this share of the AOI to be used by itself
MINIMUM_COVERAGE_PCT = 99.9

# Stop before transferring an unexpectedly large image
MAXIMUM_DOWNLOAD_MB = 100

# NAIP band order
BAND_NAMES = ('Red', 'Green', 'Blue', 'Near infrared')

# Public endpoints
STAC_URL = 'https://planetarycomputer.microsoft.com/api/stac/v1'
COURSE_DATA_URL = (
    'https://raw.githubusercontent.com/mohsennasab/python-fundamentals-hh/'
    'main/notebooks/09_naip-imagery/data'
)

# Everything this lesson writes goes in one folder
OUTPUT_DIR = Path('module9_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

width_m = AOI_BOUNDS[2] - AOI_BOUNDS[0]
height_m = AOI_BOUNDS[3] - AOI_BOUNDS[1]
print(f"Teaching AOI: {width_m:,} m wide x {height_m:,} m tall in {AOI_CRS}")
print(f"Reference year: {REFERENCE_YEAR}")
print(f"Outputs will be saved in: {OUTPUT_DIR.resolve()}")

## Part 3: Watershed and Teaching AOI

### Load the Watershed

The next cell looks for the course watershed ZIP in the usual local places first. If it isn't there (a fresh Colab session, for example), it downloads the public course copy. Only if both fail does it ask you to upload the file.

After it runs, confirm three things:

- The watershed name is Town of South Greeley.
- The reported area is about 12,017 acres.
- A CRS is printed.

In [ ]:
watershed_filename = 'NHD__Watershed_Boundaries_HUC_12_Selected.zip'
local_candidates = [
    Path(watershed_filename),
    Path('data') / watershed_filename,
    Path('notebooks/09_naip-imagery/data') / watershed_filename,
]
watershed_zip = next((path for path in local_candidates if path.exists()), None)

# A fresh Colab session has no local copy, so try the course repository
if watershed_zip is None:
    try:
        response = requests.get(f"{COURSE_DATA_URL}/{watershed_filename}", timeout=60)
        response.raise_for_status()
        watershed_zip = Path(watershed_filename)
        watershed_zip.write_bytes(response.content)
        print("Downloaded the watershed file from the course repository.")
    except requests.RequestException as error:
        print(f"Course download failed ({error}).")

# Last resort: ask for an upload
if watershed_zip is None:
    from google.colab import files
    print(f"Please upload {watershed_filename}.")
    files.upload()
    watershed_zip = Path(watershed_filename)

watersheds = gpd.read_file(f"zip://{watershed_zip}")
target_watershed = watersheds[watersheds['HUC12'].astype(str) == TARGET_HUC].copy()

if target_watershed.empty:
    raise RuntimeError(f"HUC-12 {TARGET_HUC} was not found in {watershed_zip}.")
if target_watershed.crs is None:
    raise RuntimeError("The watershed file has no CRS. Check that the ZIP includes its .prj file.")

print(f"Watershed source: {watershed_zip}")
print(f"Target watershed: {target_watershed['Name'].iloc[0]}")
print(f"Reported area: {target_watershed['AreaAcres'].iloc[0]:,.0f} acres")
print(f"Watershed CRS: {target_watershed.crs}")

### Build the Teaching AOI

Why not download NAIP for the whole watershed? At 0.6 m, the rectangle around the 12,000-acre watershed is roughly 20,000 by 12,000 pixels per band. With four bands that's close to a gigabyte in memory and several minutes of transfer, all before you've checked whether it's the right image. The teaching AOI is about 400 acres and a few megabytes.

The habit is worth keeping on real projects too: **prove the workflow on a small area, then scale up.** The advanced notebook for this module (`09_04_advanced_watershed_mosaic.ipynb`) shows how to build a full-watershed mosaic once you're ready.

The next cell creates the AOI rectangle, converts it to longitude and latitude for the catalog search, and confirms it sits inside the watershed.

In [ ]:
# The AOI as a one-row GeoDataFrame in the projected CRS
aoi = gpd.GeoDataFrame({'name': ['Teaching AOI']}, geometry=[box(*AOI_BOUNDS)], crs=AOI_CRS)
aoi_geometry = aoi.geometry.iloc[0]

# STAC searches use longitude and latitude
aoi_lonlat_bounds = aoi.to_crs('EPSG:4326').total_bounds

# Put the watershed in the same projected CRS so areas are in square meters
watershed_utm = target_watershed.to_crs(AOI_CRS)
watershed_geometry = watershed_utm.geometry.iloc[0]
inside_pct = 100 * aoi_geometry.intersection(watershed_geometry).area / aoi_geometry.area

print(f"AOI area: {aoi_geometry.area / 4046.8564224:,.0f} acres "
      f"({aoi_geometry.area / 1e6:.2f} km²)")
print(f"Watershed area: {watershed_geometry.area / 4046.8564224:,.0f} acres")
print(f"Share of the AOI inside the watershed: {inside_pct:.1f}%")
print("AOI in longitude/latitude (W, S, E, N):", np.round(aoi_lonlat_bounds, 5))

if inside_pct < 99.9:
    raise RuntimeError("The teaching AOI is not inside the watershed. Check AOI_BOUNDS and AOI_CRS.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
watershed_utm.boundary.plot(ax=ax, color='#1f4e79', linewidth=1.5)
aoi.boundary.plot(ax=ax, color='#c0392b', linewidth=2.5)

ax.annotate('Teaching AOI', xy=(AOI_BOUNDS[2], AOI_BOUNDS[3]), xytext=(10, 10),
            textcoords='offset points', color='#c0392b', fontsize=11)
ax.set_title(f"{target_watershed['Name'].iloc[0]} HUC-12 and the Module 9 teaching AOI")
ax.set_xlabel('Easting (m, UTM 13N)')
ax.set_ylabel('Northing (m, UTM 13N)')
ax.ticklabel_format(style='plain', useOffset=False)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The red rectangle is small compared with the watershed, and that's intentional. It sits in the north-central part of the HUC, where the college campus, a new subdivision, and open rangeland meet.

## Part 4: Search the NAIP Catalog

### First, a Word About Sharing a Free Service

The NAIP catalog is free and needs no account, which means everyone using it anonymously shares one pool of capacity. When requests arrive too fast, the service protects itself by answering **HTTP 429, "too many requests"** instead of data. You'll see a message like *"You have exceeded a rate limit."*

This comes up more often than you'd expect in a class, for a reason worth knowing: **Colab sends everyone's requests through a small set of shared addresses.** As far as the service can tell, thirty students in one session can look like one very busy user. You can hit the limit without having done anything wrong.

A 429 is not an error in your code, and it isn't an outage either. It means *slow down and try again*. The standard way to handle it is to wait a moment and retry, with a longer wait each time. That's called exponential backoff, and the next cell sets it up once so every request in this lesson benefits.

Two things the helper handles that are easy to miss:

- **The default retry policy doesn't cover status codes at all.** The libraries underneath `pystac-client` retry dropped connections, but a 429 is a perfectly valid HTTP response, so it's passed straight through to you. Status codes you want retried have to be listed explicitly.
- **A STAC search is sent as a POST request.** Retry policies often cover only GET, on the theory that other methods might change something on the server. A search doesn't change anything, so it's safe to retry.

In [ ]:
def open_naip_catalog():
    """Open the NAIP catalog with a session that waits out rate limits.

    Without `status_forcelist`, urllib3 retries dropped connections but not
    HTTP status codes, so a single 429 would end the search. Listing 429 makes
    the session pause and try again, and `respect_retry_after_header` lets the
    server tell us how long to wait when it sends a Retry-After header.
    """
    rate_limit_retry = Retry(
        total=5,                                    # up to five extra attempts
        backoff_factor=1.0,                         # waits about 1, 2, 4, 8, 16 seconds
        status_forcelist=[429, 500, 502, 503, 504],  # 429 is the rate limit
        allowed_methods=None,                       # retry POST searches, not just GET
        respect_retry_after_header=True,
    )
    return pystac_client.Client.open(
        STAC_URL, stac_io=StacApiIO(max_retries=rate_limit_retry))


def was_rate_limited(error):
    """True if an exception looks like an HTTP 429.

    Two shapes to catch. A 429 that comes straight back carries
    `status_code`. A 429 that survives every retry arrives instead as a
    connection error whose text mentions "too many 429 error responses",
    with no status code at all.
    """
    text = str(error).lower()
    return (getattr(error, 'status_code', None) == 429
            or 'rate limit' in text or '429' in text)


def explain_search_failure(error):
    """Turn a catalog exception into something a student can act on."""
    if was_rate_limited(error):
        return ("The catalog is rate limiting this session (HTTP 429), and waiting "
                "and retrying was not enough.\n"
                "  Nothing is wrong with your code. Colab shares a few outgoing "
                "addresses, so a busy\n"
                "  class can reach the anonymous limit together.\n"
                "  What to do: wait a minute or two, then rerun this cell. If it "
                "keeps happening, carry\n"
                "  on anyway. Part 7 uses the course copy of the image, and the rest "
                "of the lesson works.")
    return (f"The catalog search failed: {error}\n"
            "  This is usually a temporary service problem, not your code.\n"
            "  Part 7 falls back to the course copy of the image.")


print("Catalog helpers are ready. No request has been sent yet.")

### Run the Search

The search sends the AOI's bounding box and the collection name to the catalog. What comes back is a list of items (catalog cards). No imagery has moved yet.

If the service is busy, this cell can take up to about a minute while it waits and retries, so give it a moment before assuming it's stuck. Re-running the cell after a successful search reuses what it already has instead of asking again, which is the polite way to work with a shared service.

In [ ]:
# Reuse an earlier successful search instead of asking the service again
if 'naip_items' in globals() and naip_items:
    print(f"Reusing the {len(naip_items)} items found earlier in this session.")
else:
    print("Searching the NAIP catalog...")
    try:
        catalog = open_naip_catalog()
        naip_items = list(catalog.search(
            collections=['naip'], bbox=list(aoi_lonlat_bounds)).items())
        print(f"NAIP items that intersect the AOI: {len(naip_items)}")
    except Exception as error:
        # A failed search is survivable: Part 7 falls back to the course copy.
        catalog = None
        naip_items = []
        print(explain_search_failure(error))

### Build an Inventory Table

A list of item objects is hard to review, so the next function turns it into a table with only the fields that matter for an engineering decision:

- `item_id` - the stable name of the image tile
- `naip_year` and `acquisition_date` - when it was flown
- `gsd_m` - source ground sample distance
- `source_epsg` - the source CRS
- `band_count` - bands listed in the catalog
- `aoi_coverage_pct` - how much of the AOI falls inside the footprint
- `covers_aoi` - whether the footprint covers the whole AOI

Coverage is measured after reprojecting each footprint to UTM meters. Footprints in the catalog are in longitude and latitude, and degree-based areas aren't meaningful.

In [ ]:
def build_inventory(items, target_geometry, target_crs):
    """Summarize NAIP items and measure how much of the target each covers."""
    records = []
    for item in items:
        footprint = gpd.GeoSeries([shape(item.geometry)], crs='EPSG:4326')
        footprint = footprint.to_crs(target_crs).iloc[0]
        covered_area = footprint.intersection(target_geometry).area
        image_asset = item.assets.get('image')
        band_list = []
        if image_asset is not None:
            band_list = (image_asset.extra_fields.get('eo:bands')
                         or image_asset.extra_fields.get('bands') or [])
        records.append({
            'item_id': item.id,
            'naip_year': int(item.properties['naip:year']),
            'acquisition_date': item.datetime.date().isoformat(),
            'gsd_m': float(item.properties['gsd']),
            'source_epsg': (item.properties.get('proj:epsg')
                            or item.properties.get('proj:code')),
            'band_count': len(band_list),
            'aoi_coverage_pct': round(100 * covered_area / target_geometry.area, 1),
            'covers_aoi': bool(footprint.covers(target_geometry)),
        })

    columns = ['item_id', 'naip_year', 'acquisition_date', 'gsd_m', 'source_epsg',
               'band_count', 'aoi_coverage_pct', 'covers_aoi']
    # Explicit types keep the table usable even when the search returned nothing
    inventory = pd.DataFrame(records, columns=columns).astype(
        {'naip_year': int, 'gsd_m': float, 'aoi_coverage_pct': float, 'covers_aoi': bool})
    # Newest first; item_id breaks ties so the order is repeatable
    return inventory.sort_values(['acquisition_date', 'item_id'],
                                 ascending=False).reset_index(drop=True)


inventory = build_inventory(naip_items, aoi_geometry, AOI_CRS)
inventory

Read the table before moving on. For this AOI you should see five acquisitions: 2012, 2015, 2017, 2019, and 2022. A few things to notice:

- **GSD changes over time.** 1 m in 2012 and 2017, 0.5 m in 2015, 0.6 m in 2019 and 2022. Finer isn't automatically better for comparisons, as Lesson 2 will show.
- **Dates are specific.** The 2022 image over this AOI was flown on June 21. Another 2022 tile in Wyoming could have been flown weeks earlier or later.
- **Every row covers 100 percent of the AOI.** The AOI was placed inside a single NAIP tile on purpose. Real project areas often straddle tile edges, which is what Part 11 is about.

**Question to ask your AI assistant:** *"This inventory has five rows for five different years. Could two rows from the same year ever both be needed for one AOI? What would that look like in the coverage column?"*

## Part 5: Choose an Acquisition

### The Selection Rule

The rule has two parts, applied in this order:

1. **Keep only items that cover the whole AOI.** This comes first because a partial image is never the right answer for a single-image workflow, no matter how recent it is.
2. **Of those, take the one from the reference year.** If there's more than one (rare for a small AOI), take the newest.

The cell also lists every year with full coverage. On a real project that list is often the first thing you'd share with the team: "We have complete imagery for 2012, 2015, 2017, 2019, and 2022."

In [ ]:
full_coverage = inventory.loc[
    (inventory['aoi_coverage_pct'] >= MINIMUM_COVERAGE_PCT) & inventory['covers_aoi']
]
print("Years with a single image covering the whole AOI:",
      sorted(full_coverage['naip_year'].unique().tolist(), reverse=True))

candidates = full_coverage.loc[full_coverage['naip_year'] == REFERENCE_YEAR]

if candidates.empty:
    selected_row = None
    selected_item = None
    print(f"\nNo live {REFERENCE_YEAR} image covers the AOI (or the catalog was unavailable).")
    print("Part 7 will use the course copy of the reference image.")
else:
    selected_row = candidates.iloc[0]
    selected_item = next(item for item in naip_items if item.id == selected_row['item_id'])
    print(f"\nSelected item:     {selected_row['item_id']}")
    print(f"Acquisition date:  {selected_row['acquisition_date']}")
    print(f"Source GSD:        {selected_row['gsd_m']} m")
    print(f"Source CRS:        {selected_row['source_epsg']}")
    print(f"AOI coverage:      {selected_row['aoi_coverage_pct']}%")

You should see item `wy_m_4110458_ne_13_060_20220621`, acquired 2022-06-21, at 0.6 m. The item name is worth decoding once: `wy` is the state, `4110458` is the USGS quadrangle, `ne` is the northeast quarter, `13` is the UTM zone, `060` is 0.60 m, and `20220621` is the flight date.

## Part 6: Estimate the Download Before You Start

Array size is `columns x rows x bands x bytes per value`. NAIP uses four one-byte bands. The next cell estimates the AOI download at the source GSD and compares it with the whole watershed at the same resolution. It stops if the AOI estimate is larger than `MAXIMUM_DOWNLOAD_MB`.

In [ ]:
source_gsd = float(selected_row['gsd_m']) if selected_row is not None else 0.6

def estimate_mb(bounds, pixel_size, band_count=4, bytes_per_value=1):
    """Uncompressed size of a raster covering `bounds` at `pixel_size`."""
    columns = math.ceil((bounds[2] - bounds[0]) / pixel_size)
    rows = math.ceil((bounds[3] - bounds[1]) / pixel_size)
    return columns, rows, columns * rows * band_count * bytes_per_value / 1024 ** 2

aoi_cols, aoi_rows, aoi_mb = estimate_mb(AOI_BOUNDS, source_gsd)
ws_cols, ws_rows, ws_mb = estimate_mb(watershed_utm.total_bounds, source_gsd)

print(f"At the source GSD of {source_gsd} m:")
print(f"  Teaching AOI: {aoi_cols:,} x {aoi_rows:,} pixels, about {aoi_mb:,.0f} MB uncompressed")
print(f"  Whole watershed rectangle: {ws_cols:,} x {ws_rows:,} pixels, about {ws_mb:,.0f} MB")

if aoi_mb > MAXIMUM_DOWNLOAD_MB:
    raise RuntimeError(f"The AOI would need about {aoi_mb:.0f} MB. Shrink the AOI first.")
print("\nSize check passed.")

The AOI is about 18 MB before compression. The watershed rectangle would be more than fifty times larger. That difference is why we test on a small area.

**Question to ask your AI assistant:** *"If NAIP resolution improved from 0.6 m to 0.3 m, how much bigger would my download get, and why isn't the answer just 'twice as big'?"*

## Part 7: Download Only the AOI

### How the Read Works

The 2022 source tile covers a whole quarter quadrangle, about 6 km by 7.5 km. We don't want all of it. The helper below:

1. **Signs** the image asset. Planetary Computer's catalog is public, but reading the image file needs a short-lived token. `planetary_computer.sign()` adds it to the URL. The signed URL stays inside the function and is never printed or saved. This step has its own rate limit, but the `planetary-computer` package already retries 429s for you and caches the token, so signing several items in a row costs one request.
2. **Opens** the remote COG with Rasterio.
3. **Converts** the AOI bounds to a pixel window, rounded outward to whole source pixels. Because we don't resample, the pixel values in our file are the source values.
4. **Reads** just that window of all four bands.

A second helper writes the GeoTIFF with an internal valid-data mask, band names, and provenance tags: item ID, acquisition date, source GSD, and access date. A third downloads the course copy if the live read fails.

In [ ]:
def read_aoi_window(item, bounds):
    """Read `bounds` from a NAIP item's image on the item's own pixel grid.

    The signed URL is created here and never printed, returned, or saved.
    """
    signed_href = planetary_computer.sign(item.assets['image'].href)
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
        with rasterio.open(signed_href) as source:
            if source.count < 4:
                raise RuntimeError(f"Expected four bands, found {source.count}.")
            window = from_bounds(*bounds, transform=source.transform)

            # Round outward to whole source pixels so nothing is resampled
            col_start = math.floor(window.col_off)
            row_start = math.floor(window.row_off)
            col_stop = math.ceil(window.col_off + window.width)
            row_stop = math.ceil(window.row_off + window.height)
            if (col_start < 0 or row_start < 0
                    or col_stop > source.width or row_stop > source.height):
                raise RuntimeError("The AOI extends past the edge of this image.")
            window = Window(col_start, row_start,
                            col_stop - col_start, row_stop - row_start)

            bands = source.read([1, 2, 3, 4], window=window)
            transform = source.window_transform(window)
            crs = source.crs
    return bands, transform, crs


def write_naip_geotiff(path, bands, transform, crs, item):
    """Write a four-band NAIP crop with a valid-data mask and provenance tags."""
    # NAIP has no NoData value; all-zero pixels are treated as missing
    valid = np.any(bands != 0, axis=0)
    profile = {
        'driver': 'GTiff', 'count': 4, 'dtype': 'uint8',
        'width': bands.shape[2], 'height': bands.shape[1],
        'crs': crs, 'transform': transform,
        'compress': 'deflate', 'predictor': 2, 'tiled': True,
        'blockxsize': 256, 'blockysize': 256, 'interleave': 'pixel',
    }
    with rasterio.Env(GDAL_TIFF_INTERNAL_MASK=True):
        with rasterio.open(path, 'w', **profile) as destination:
            destination.write(bands)
            destination.write_mask(valid.astype('uint8') * 255)
            for band_number, band_name in enumerate(BAND_NAMES, start=1):
                destination.set_band_description(band_number, band_name)
            destination.update_tags(
                source_program='USDA National Agriculture Imagery Program',
                source_host='Microsoft Planetary Computer',
                stac_collection='naip',
                source_item_id=item.id,
                naip_year=str(item.properties['naip:year']),
                acquisition_date=item.datetime.date().isoformat(),
                source_gsd_m=str(item.properties['gsd']),
                source_epsg=str(item.properties.get('proj:epsg')
                                or item.properties.get('proj:code')),
                access_date=datetime.date.today().isoformat(),
                processing='AOI window read on the source pixel grid; '
                           'no reprojection or resampling',
                license='Public domain (USDA Farm Service Agency)',
            )


def get_course_copy(filename, destination):
    """Copy a course data file from a local clone, or download it from GitHub."""
    for candidate in [Path('data') / filename,
                      Path('notebooks/09_naip-imagery/data') / filename]:
        if candidate.exists():
            shutil.copyfile(candidate, destination)
            return f"local course file ({candidate})"
    response = requests.get(f"{COURSE_DATA_URL}/{filename}", timeout=120)
    response.raise_for_status()
    destination.write_bytes(response.content)
    return "course repository download"


print("Download helpers are ready. Nothing has been downloaded yet.")

### Run the Download

This is the first cell that transfers image data. It usually takes 10 to 30 seconds on Colab.

Read the `Image source` line when it finishes. `Live read from Planetary Computer` means the live workflow worked end to end. If the live read fails, the cell says why and switches to the course copy. The course copy was made with the same helper from the same item, so everything downstream behaves the same. Either way, Part 8 checks the file.

In [ ]:
reference_path = OUTPUT_DIR / f'naip_{REFERENCE_YEAR}_teaching_aoi.tif'
image_source = None

if selected_item is not None:
    try:
        bands, transform, crs = read_aoi_window(selected_item, AOI_BOUNDS)
        write_naip_geotiff(reference_path, bands, transform, crs, selected_item)
        image_source = 'Live read from Planetary Computer'
    except (rasterio.errors.RasterioError, requests.RequestException,
            RuntimeError, OSError) as error:
        print(f"The live read failed: {error}")
        print("(Timeouts, rate limits, and 5xx errors come from the service, "
              "not from your notebook.)")

if image_source is None:
    print("Switching to the course copy of the same image...")
    image_source = get_course_copy(reference_path.name, reference_path)

print(f"Image source: {image_source}")
print(f"Saved: {reference_path} ({reference_path.stat().st_size / 1024 ** 2:.1f} MB on disk)")

## Part 8: Inspect the File Before You Trust It

A file that opens isn't necessarily the file you wanted. The next cell reopens the GeoTIFF from disk and runs a checklist:

| Check | Why it matters |
|---|---|
| Four `uint8` bands named Red, Green, Blue, Near infrared | Composites and NDVI depend on band order |
| CRS is EPSG:26913 | Everything downstream assumes UTM meters |
| Raster bounds contain the AOI | Catches partial or shifted crops |
| Pixel size matches the source GSD | Confirms nothing was resampled |
| Valid pixels cover the AOI | Catches missing strips inside the bounds |
| Provenance tags are present | Date and item ID must travel with the file |
| No access token in the tags | Temporary credentials shouldn't be stored |

The cell prints a PASS/FAIL table and stops if anything fails.

In [ ]:
REQUIRED_TAGS = ['source_item_id', 'naip_year', 'acquisition_date', 'source_gsd_m',
                 'source_epsg', 'access_date', 'processing', 'license']
TOKEN_MARKERS = ['sig=', 'se=', 'sp=', 'sv=', 'skoid=']

with rasterio.open(reference_path) as source:
    profile = source.profile
    raster_bounds = source.bounds
    pixel_size = source.res
    descriptions = source.descriptions
    tags = source.tags()
    valid_fraction = (source.dataset_mask() > 0).mean()

tolerance = 0.01
checks = [
    ('Driver is GTiff', profile['driver'] == 'GTiff', profile['driver']),
    ('Four bands', profile['count'] == 4, profile['count']),
    ('Unsigned 8-bit values', profile['dtype'] == 'uint8', profile['dtype']),
    ('Band order R, G, B, NIR', tuple(descriptions) == BAND_NAMES, descriptions),
    ('CRS is the AOI CRS', profile['crs'].to_string() == AOI_CRS, profile['crs'].to_string()),
    ('Raster bounds contain the AOI',
     raster_bounds.left <= AOI_BOUNDS[0] + tolerance
     and raster_bounds.bottom <= AOI_BOUNDS[1] + tolerance
     and raster_bounds.right >= AOI_BOUNDS[2] - tolerance
     and raster_bounds.top >= AOI_BOUNDS[3] - tolerance,
     tuple(round(value, 2) for value in raster_bounds)),
    ('Pixel size equals source GSD',
     math.isclose(pixel_size[0], float(tags.get('source_gsd_m', 'nan')), abs_tol=0.001),
     f"{pixel_size[0]} m vs GSD {tags.get('source_gsd_m')} m"),
    ('Valid pixels cover the AOI', valid_fraction >= 0.999, f"{valid_fraction:.2%}"),
    ('Provenance tags present', all(tag in tags for tag in REQUIRED_TAGS),
     [tag for tag in REQUIRED_TAGS if tag not in tags] or 'all present'),
    ('No access token stored',
     not any(marker in ' '.join(tags.values()).lower() for marker in TOKEN_MARKERS),
     'clean'),
]

check_table = pd.DataFrame(checks, columns=['check', 'passed', 'value'])
check_table['status'] = np.where(check_table['passed'], 'PASS', 'FAIL')
display(check_table[['check', 'status', 'value']])

if not check_table['passed'].all():
    raise RuntimeError("The image failed at least one check. Review the table above.")

print(f"\nAll checks passed.")
print(f"Source item: {tags['source_item_id']}, acquired {tags['acquisition_date']}")
print(f"Size: {profile['width']:,} x {profile['height']:,} pixels at {pixel_size[0]} m")

Two details in that table are worth a closer look:

- **The bounds are a little larger than the AOI.** The west edge is 517,749.6 instead of 517,750. The source pixel grid doesn't happen to line up with our round-number corners, so the helper rounded outward to whole pixels. That's the price of not resampling, and it's the right trade.
- **The pixel size is 0.6 m, the same as the source GSD.** In Lesson 2 you'll put two years on a shared 1 m grid, and those two numbers will differ on purpose.

**Question to ask your AI assistant:** *"My GeoTIFF's bounds are 0.4 m larger than the AOI I asked for. Is that a problem? What would I have to do differently to hit the AOI corners exactly, and what would that cost me?"*

## Part 9: First Look

### A True-Color Preview

NAIP values are 0 to 255, but most pixels fall in a narrower range, so a raw display looks flat. The helper below stretches each band between its 2nd and 98th percentiles for display only. The GeoTIFF isn't changed.

The figure uses the raster's real-world extent, so the axes are UTM coordinates you could check against a GIS.

In [ ]:
def percentile_stretch(bands, lower=2, upper=98):
    """Stretch a (3, rows, cols) array to 0-1 for display. The data are not changed."""
    display_array = np.zeros(bands.shape, dtype='float32')
    for index in range(bands.shape[0]):
        band = bands[index].astype('float32')
        low, high = np.percentile(band, [lower, upper])
        display_array[index] = np.clip((band - low) / max(high - low, 1), 0, 1)
    return np.moveaxis(display_array, 0, -1)


with rasterio.open(reference_path) as source:
    rgb = source.read([1, 2, 3])
    extent = plotting_extent(source)
    acquisition_date = source.tags()['acquisition_date']

fig, ax = plt.subplots(figsize=(8, 11))
ax.imshow(percentile_stretch(rgb), extent=extent)
aoi.boundary.plot(ax=ax, color='yellow', linewidth=1)
ax.set_title(f"NAIP true color, acquired {acquisition_date}")
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
ax.ticklabel_format(style='plain', useOffset=False)
plt.tight_layout()

quicklook_path = OUTPUT_DIR / f'naip_{REFERENCE_YEAR}_quicklook.png'
fig.savefig(quicklook_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {quicklook_path}")

Orient yourself before moving on:

- **North:** open rangeland, a graded soil area, and a long pond (the north pond).
- **Center:** the college campus. You'll see the athletic field, parking lots, rooftops, and a smaller pond on the west side.
- **South:** a new subdivision on the left and graded construction ground on the right. In June 2022 houses were still going up.

### The Four Bands, One at a Time

Composites mix three bands together, which makes it easy to forget what each one holds. The next figure shows each band on the same 0 to 255 gray scale, so brightness is directly comparable across panels.

Look at the athletic field and the two ponds in each panel. The contrast between the red and near-infrared panels is the basis of the color-infrared views and NDVI in Lesson 2.

In [ ]:
with rasterio.open(reference_path) as source:
    all_bands = source.read()

fig, axes = plt.subplots(1, 4, figsize=(18, 6.5))
for axis, band, name in zip(axes, all_bands, BAND_NAMES):
    image = axis.imshow(band, cmap='gray', vmin=0, vmax=255, extent=extent)
    axis.set_title(f"Band: {name}\nmean value {band.mean():.0f}")
    axis.set_xticks([])
    axis.set_yticks([])
fig.colorbar(image, ax=axes, shrink=0.6, label='Pixel value (0-255)')
plt.show()

What to notice:

- The **athletic field and trees are bright in NIR** but dark in red. Healthy vegetation absorbs red light for photosynthesis and reflects near infrared.
- **Water is dark in every band**, darkest of all in NIR.
- **Rooftops and graded soil are bright in all four bands**, which is why telling them apart takes more than one band.
- The **dry rangeland is a mid-gray almost everywhere**. In late June this grass is partly cured, so it doesn't stand out in NIR the way the irrigated field does. Keep that in mind for Lesson 3.

## Part 10: Export a Source Inventory

The inventory CSV records every acquisition that touches the AOI, which one you selected, and where the image came from. It's the file you'd attach to a project memo so someone else can reproduce your choice.

The cell checks the text for token-like strings before writing. The inventory stores item IDs, which are stable, and never signed URLs, which expire.

In [ ]:
inventory_export = inventory.copy()
inventory_export['selected'] = inventory_export['item_id'] == tags['source_item_id']
inventory_export['stac_collection'] = 'naip'
inventory_export['source_host'] = 'Microsoft Planetary Computer'
inventory_export['access_date'] = datetime.date.today().isoformat()
inventory_export['output_file'] = np.where(inventory_export['selected'], reference_path.name, '')

export_text = inventory_export.to_csv(index=False).lower()
if any(marker in export_text for marker in TOKEN_MARKERS):
    raise RuntimeError("A token-like string was found in the inventory. Nothing was saved.")

inventory_path = OUTPUT_DIR / 'naip_source_inventory.csv'
inventory_export.to_csv(inventory_path, index=False)
print(f"Saved: {inventory_path} ({len(inventory_export)} rows)")
if inventory_export.empty:
    print("The live catalog was unavailable, so the inventory is empty. "
          "The image tags still record its source item and date.")
inventory_export

## Part 11: Practice Debugging - When "The Newest Image" Isn't the Right Image

Everything so far worked because the teaching AOI sits inside one tile. Real project areas don't cooperate like that. Here's a second test box, 1 km by 600 m, placed on purpose so it straddles the boundary between two NAIP tiles.

The next cell searches for it and builds the same inventory table. Look at `aoi_coverage_pct` and `covers_aoi` before you run the cell after it.

In [ ]:
# A test box that crosses a tile boundary on purpose (UTM 13N meters)
EDGE_TEST_BOUNDS = (515300, 4549000, 516300, 4549600)
edge_box = box(*EDGE_TEST_BOUNDS)
edge_lonlat = gpd.GeoSeries([edge_box], crs=AOI_CRS).to_crs('EPSG:4326').total_bounds

edge_items = []
if catalog is None:
    print("The catalog is unavailable, so this exercise needs to wait for the live service.")
else:
    try:
        edge_items = list(catalog.search(
            collections=['naip'], bbox=list(edge_lonlat)).items())
        print(f"NAIP items that intersect the test box: {len(edge_items)}")
    except Exception as error:
        print(explain_search_failure(error))

edge_inventory = build_inventory(edge_items, edge_box, AOI_CRS)
edge_inventory

### The Buggy Cell

The next cell uses a selection rule that sounds reasonable: *take the newest image the search returned.* It runs without an error and prints a sensible-looking item ID. Run it and look carefully at the preview.

> **This cell contains an intentional bug.** It's here for practice. Nothing it does affects the reference image you saved in Part 7.

In [ ]:
# ---------------------------------------------------------------
# PRACTICE DEBUGGING: THIS CELL CONTAINS AN INTENTIONAL BUG.
# It runs without an error. Your job is to explain what's wrong.
# ---------------------------------------------------------------
preview = None
if edge_inventory.empty:
    print("No items to choose from. Rerun Part 11 when the catalog is available.")
else:
    # Selection rule: take the newest image returned by the search
    wrong_row = edge_inventory.iloc[0]
    wrong_item = next(item for item in edge_items if item.id == wrong_row['item_id'])
    print(f"Selected: {wrong_row['item_id']} (acquired {wrong_row['acquisition_date']})")

    # A quick 5 m preview of what this one image holds inside the test box
    preview_res = 5
    preview_shape = (3, round((EDGE_TEST_BOUNDS[3] - EDGE_TEST_BOUNDS[1]) / preview_res),
                     round((EDGE_TEST_BOUNDS[2] - EDGE_TEST_BOUNDS[0]) / preview_res))
    try:
        signed_href = planetary_computer.sign(wrong_item.assets['image'].href)
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open(signed_href) as source:
                window = from_bounds(*EDGE_TEST_BOUNDS, transform=source.transform)
                preview = source.read([1, 2, 3], window=window, out_shape=preview_shape,
                                      boundless=True, masked=True)
    except Exception as error:
        # The picture is nice to have, but the coverage column already tells the story
        preview = None
        print(f"\nCould not read the preview image: {error}")
        print("The exercise still works. Compare the aoi_coverage_pct and covers_aoi "
              "columns in the table above.")

if not edge_inventory.empty and preview is not None:
    wrong_footprint = gpd.GeoSeries([shape(wrong_item.geometry)], crs='EPSG:4326').to_crs(AOI_CRS)
    edge_extent = (EDGE_TEST_BOUNDS[0], EDGE_TEST_BOUNDS[2], EDGE_TEST_BOUNDS[1], EDGE_TEST_BOUNDS[3])

    fig, ax = plt.subplots(figsize=(10, 6.5))
    ax.imshow(percentile_stretch(preview.filled(0)), extent=edge_extent)
    wrong_footprint.boundary.plot(ax=ax, color='cyan', linewidth=2)
    gpd.GeoSeries([edge_box], crs=AOI_CRS).boundary.plot(ax=ax, color='red', linewidth=2)
    ax.set_xlim(EDGE_TEST_BOUNDS[0] - 100, EDGE_TEST_BOUNDS[2] + 100)
    ax.set_ylim(EDGE_TEST_BOUNDS[1] - 100, EDGE_TEST_BOUNDS[3] + 100)
    ax.ticklabel_format(style='plain', useOffset=False)
    ax.set_title("Red: test box   Cyan: selected image footprint")
    plt.show()

### Your Task

Part of that preview is black: there's no imagery there at all. Before you change any code:

1. **Describe the symptom.** What part of the test box has no imagery? Does it match anything in the edge inventory table?
2. **Find the faulty assumption.** What did the selection rule assume about the newest image? Which column in `edge_inventory` would have caught it?
3. **Decide what the right answer is.** Look at the whole table. Is there *any* single image that covers this box? If not, what should a well-behaved selection step do?

**Try this with your AI assistant:**

*"My selected NAIP item intersects my test box, but the preview has a strip with no imagery. Here's my inventory table: [paste the edge_inventory table]. What assumption in the selection step is wrong? Suggest a check to add before I change any code."*

Then write your fix in the cell below. A good fix does two things: it never hands back a partial image, and it tells the user clearly what to do instead.

In [ ]:
# YOUR FIX: choose from edge_inventory with a rule that guarantees full coverage.
#
# Hints:
# 1. You already have a column that answers "does this image cover the box?"
# 2. Decide what happens when no row qualifies. An empty result is a valid
#    outcome, and the message you print is part of the fix.
# 3. Compare your rule with the one used in Part 5.

**Going further (optional):** Both 2022 tiles here were flown on the same date, so a same-date mosaic would be a defensible way to cover this box. The advanced notebook `09_04_advanced_watershed_mosaic.ipynb` builds exactly that kind of mosaic for the whole watershed, using raw HTTP requests, pagination, and year-by-year coverage checks. It's a good read if you want to see what `pystac-client` handles for you behind the scenes.

## Engineering Interpretation and Limitations

### What This Image Supports

- Site orientation and visual review: roads, rooftops, ponds, graded ground, and vegetation
- A dated, traceable background for maps and memos
- The input for visual comparison (Lesson 2) and screening classification (Lesson 3)

### What It Doesn't Establish

- **Elevation, slope, or drainage connectivity.** An image can't tell you which way water flows.
- **Impervious percentage.** A roof looks like a roof, but connected impervious area needs drainage information.
- **Conditions on a storm date.** It shows June 21, 2022, and nothing else.
- **Survey-grade positions.** NAIP's horizontal accuracy spec is a few meters. Use project control for measurements.

### Cautions Worth Remembering

1. **"Newest" isn't a coverage test.** Always check the footprint against the AOI.
2. **Acquisition date belongs to each image.** Record it for every file.
3. **GSD and output pixel size are different metadata.** Keep both.
4. **Item IDs are provenance. Signed URLs are temporary credentials.** Store the first, never the second.
5. **NAIP values are for viewing, not physics.** They're adjusted for appearance, so don't treat them as calibrated reflectance.

## Troubleshooting

| Problem | Likely cause | What to try |
|---|---|---|
| Package install fails | Temporary network issue in Colab | Rerun the install cell; restart the runtime if imports still fail |
| "You have exceeded a rate limit" (HTTP 429) | Too many anonymous requests from your address, often shared with other Colab users | Wait a minute and rerun the search cell. The notebook already retries; if it persists, keep going, because Part 7 uses the course copy |
| Catalog search fails or times out | Planetary Computer busy or down | Continue; Part 7 uses the course copy. Rerun later for the live path |
| Inventory is empty | Search failed, or the AOI is outside NAIP coverage | Print `aoi_lonlat_bounds` and confirm they're in Wyoming |
| No reference-year image covers the AOI | AOI moved onto a tile edge | Move the AOI inside one tile, or pick another full-coverage year |
| Live read fails with a 403 or "not recognized as a supported file format" | Access token expired or service error | Rerun the download cell; it signs a fresh URL each time |
| Image source says "course repository download" | The live read failed, and the notebook fell back | Fine for the lesson; the file passes the same checks |
| A check fails | Wrong file, partial crop, or band order changed | Read the `value` column in the check table |
| Preview in Part 11 has no blank area | Catalog tiles or ordering changed | Compare the coverage column; the lesson is the same |

## Practice Exercises

### Exercise 1: Pick an Older Full-Coverage Year

Set `REFERENCE_YEAR` to 2015 and rerun Parts 5 through 9. The 2015 imagery has a 0.5 m GSD. Record the acquisition date, the source GSD, the pixel dimensions, and the file size. Then compare the Part 6 size estimate with the 2022 estimate.

Change `REFERENCE_YEAR` back to 2022 when you're done, because Lessons 2 and 3 use the 2022 image.

In [ ]:
# EXERCISE 1: Download an older full-coverage year
# Your code here.
#
# Steps:
# 1. Set REFERENCE_YEAR = 2015 in the settings cell (or here, then rerun Part 5).
# 2. Rerun Parts 5 to 9. The output file name includes the year.
# 3. Record acquisition date, GSD, width x height, and file size.
# 4. Set REFERENCE_YEAR back to 2022.

### Exercise 2: Plan Before You Download

Using `estimate_mb()`, build a small table of estimated download size for the teaching AOI and for the whole watershed rectangle at 0.3 m, 0.6 m, 1 m, and 2 m. Don't download anything. The point is to make the processing decision first.

In [ ]:
# EXERCISE 2: Size planning table
# Your code here.
#
# Suggested pattern:
# rows = []
# for pixel_size in [0.3, 0.6, 1.0, 2.0]:
#     cols, nrows, mb = estimate_mb(AOI_BOUNDS, pixel_size)
#     ...
# pd.DataFrame(rows)

### Challenge: Move the AOI

Pick a different 1 km² area inside the watershed, perhaps over the residential area west of Highway 85. Rerun the search and inventory. Does a single tile still cover your AOI in every year? If not, which years fail, and why?

Suggested prompt:

*"Help me adapt this notebook so I can pass in any small AOI in UTM coordinates and get back a table of years where one NAIP item fully covers it. Keep the coverage check in projected meters, and don't return partial images."*

In [ ]:
# CHALLENGE: Try a different AOI
# Your code here.
#
# Steps:
# 1. Choose new AOI corners in EPSG:26913 meters inside the watershed.
# 2. Rebuild the AOI GeoDataFrame and its longitude/latitude bounds.
# 3. Search, build the inventory, and list the years with full coverage.

### Download Your Outputs

In Colab, the next cell downloads the three lesson outputs through your browser. In a local Jupyter session it prints their paths instead.

In [ ]:
output_files = [reference_path, inventory_path, quicklook_path]

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

for output_file in output_files:
    if not output_file.exists():
        raise FileNotFoundError(output_file)
    if colab_files is not None:
        colab_files.download(str(output_file))
    else:
        print(f"Output ready: {output_file.resolve()}")

## That's Lesson 1 Done!

You searched a public imagery catalog, compared five acquisitions, picked one that covers the whole study area, and pulled a 1.7 km² window out of a much larger cloud-hosted file without resampling a pixel. Then you checked the file instead of trusting it, and you found a selection rule that fails without raising an error. That kind of failure is easy to miss on a real project.

### Key Takeaways

- A catalog search returns records. Imagery only moves when you open an asset.
- "Intersects" and "covers" are different questions. Ask the second one.
- Acquisition date, GSD, and item ID belong in every file's provenance. Signed URLs never do.
- Estimate size before you download, and test on a small area first.
- A file that opens isn't necessarily a file that's right. Check CRS, bounds, bands, resolution, and valid pixels.

### Next: Lesson 2

In `09_02_visualize_compare_naip.ipynb` you'll bring in the 2012 image of the same area, put both years on a common grid, build true-color and color-infrared views, and work out which differences are real change and which only look like it.

### Official Resources

- [NAIP on Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/dataset/naip)
- [Planetary Computer: reading data from the STAC API](https://planetarycomputer.microsoft.com/docs/quickstarts/reading-stac/)
- [Planetary Computer: access tokens (SAS)](https://planetarycomputer.microsoft.com/docs/concepts/sas/)
- [USDA FSA NAIP imagery program](https://www.fsa.usda.gov/programs-and-services/aerial-photography/imagery-programs/naip-imagery/)
- [GeoAI: download NAIP imagery](https://opengeoai.org/examples/download_naip/)
- [Earth Data Science: NAIP multiband imagery in Python](https://earthdatascience.org/courses/use-data-open-source-python/multispectral-remote-sensing/intro-naip/)